# Generate Daily Modeling Dataset

**Prerequisites**
1. `GEE_local_HSA_Daily_Climate.ipynb` completed and files downloaded to
   `{OUT_DIR}/DRIVE_CLIMATE_BY_HSA_DOWNLOAD_DAILY/`
2. `data/INF_patient_visits.csv` present

**Outputs**
- `{OUT_DIR}/INF_footprint_daily_diarrheal.csv` — daily case counts per HSA
- `{OUT_DIR}/modeling/INF_footprint_daily_modeling_dataset.csv` — merged panel


In [ ]:
# ── BOUNDARY VERSION SELECTOR ──────────────────────────────────────────────
# Choose which HSA boundary bundle to use for this analysis.
# Must match a bundle produced by HSA_FINAL.ipynb.
#   'v6'  — original greedy algorithm (no post-selection corrections)
#   'v7'  — + anchor upgrade/demotion + major-orphan promotion
#   'v8'  — + satellite bubble boundaries
BOUNDARY_VERSION = "v7"   # change as needed
# ────────────────────────────────────────────────────────────────────────────


In [ ]:
# Config
from pathlib import Path
import os
import subprocess, sys

PIPELINE_VERSION = os.environ.get("PIPELINE_VERSION", "v7")
BASE_DIR = Path(".")
OUT_DIR = Path(os.environ.get("HSA_OUT_DIR", os.environ.get("PIPELINE_OUT_DIR", f"out_{PIPELINE_VERSION}")))
MODELING_DIR = OUT_DIR / "modeling"
print("Working directory:", BASE_DIR.resolve())
print("Output directory:", OUT_DIR)


In [ ]:
# STEP 1 — Daily disease counts
print("=" * 60)
print("STEP 1: Daily diarrheal counts")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "generate_daily_disease_counts.py", "--out-dir", str(OUT_DIR)],
    capture_output=False,
)
if result.returncode != 0:
    raise RuntimeError("generate_daily_disease_counts.py failed")
print("Done.")


In [ ]:
# STEP 2 — Assemble modeling dataset
print("=" * 60)
print("STEP 2: Assemble daily modeling dataset")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "prepare_daily_modeling_dataset.py", "--out-dir", str(OUT_DIR)],
    capture_output=False,
)
if result.returncode != 0:
    raise RuntimeError("prepare_daily_modeling_dataset.py failed")
print("Done.")


In [ ]:
# STEP 3 — Verify output
import pandas as pd

df = pd.read_csv(MODELING_DIR / "INF_footprint_daily_modeling_dataset.csv",
                 parse_dates=["date"])
print(f"Shape:       {df.shape}")
print(f"HSAs:        {df['hsa_id'].nunique()}")
print(f"Date range:  {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Columns:     {list(df.columns[:12])} ...")
print()
print("Non-zero outcome days:", (df["diarrheal_count"] > 0).sum())
print("Zero-count fraction:  ", f"{(df['diarrheal_count']==0).mean()*100:.1f}%")
print()
print("Climate coverage (non-null P_precip):",
      f"{df['P_precip'].notna().sum():,} / {len(df):,}")
print()
print("Lag columns present (sample):")
lag_cols = [c for c in df.columns if "lag" in c][:8]
print(" ", lag_cols)
